# 5. Naming a theme, and refusing to

*Part of the [broadlisten](https://github.com/InteractionDataLab/broad-listening/tree/main/code)
stage-by-stage examples. Every number and figure below was produced by running
this notebook on the synthetic corpus in `broadlisten.sample`, which is
generated from templates and contains no participant data.*

A language model always returns a name for a group of text. It does not tell you
when the group had no shared topic. Every check in this stage is therefore
allowed to fail, and a name that fails one is **withheld** rather than printed.

**This notebook runs without a GPU.** It uses a stub generator that fabricates
nothing: it returns `insufficient_evidence`, which is a real answer the prompt
asks for and the gate knows how to handle. So what you see below is the refusal
path, running for real.

In [1]:
import numpy as np
from broadlisten.cluster import hdbscan_cluster
from broadlisten.sample import sample_corpus, sample_embeddings

corpus = sample_corpus(seed=0)
texts = corpus.text.astype(str).tolist()
vectors = sample_embeddings(corpus.planted_theme.tolist(), seed=0)
labels, info = hdbscan_cluster(vectors, min_cluster_size=15, verbose=False)
print(f"{info['n_clusters']} themes to name")

5 themes to name


## What the model is actually shown

Not the whole group. Representative members chosen by MMR so they are not
near-duplicates, the most atypical members so heterogeneity is visible, and
keywords that distinguish the group from the rest of the corpus.

In [2]:
from broadlisten.label import (DEFAULT_GATE, build_naming_prompt,
                               farthest_select, logodds_keywords, mmr_select)
from sklearn.feature_extraction.text import CountVectorizer

vec = CountVectorizer(min_df=2)
matrix = vec.fit_transform(texts)
vocab = np.asarray(vec.get_feature_names_out())
members = [i for i, k in enumerate(labels) if k == 0]

keywords = logodds_keywords(np.asarray(matrix[members].sum(axis=0)).ravel(),
                            np.asarray(matrix.sum(axis=0)).ravel(), vocab, top=8)
rep = mmr_select(vectors, members, 4, lam=DEFAULT_GATE.mmr_lambda)
far = [i for i in farthest_select(vectors, members, 2) if i not in rep]

print("distinguishing keywords:", [w for w, _ in keywords])
prompt = build_naming_prompt({"n": len(members), "median_words": 8,
                              "min_words": 3, "max_words": 20,
                              "question_mix": "synthetic"},
                             [texts[i] for i in rep], [texts[i] for i in far],
                             keywords)
print(prompt[:900])

distinguishing keywords: ['weeks', 'difficult', 'through', 'each', 'supported', 'close', 'quickly', 'team']
GROUP STATISTICS
  responses in group : 114
  median length      : 8 words
  length range       : 3-20 words
  survey questions   : synthetic

DISTINGUISHING KEYWORDS (log-odds vs the rest of the corpus)
  weeks, difficult, through, each, supported, close, quickly, team

REPRESENTATIVE RESPONSES
[1] "the group was welcoming from the first day"
[2] "the team became close very quickly"
[3] "we supported each other through the difficult weeks"
[4] "we supported each other through the difficult weeks"

MOST ATYPICAL MEMBERS OF THE SAME GROUP
[5] "Honestly, the team became close very quickly"
[6] "Это был потрясающий опыт"

Step 1. Read every response. Do they share a recognisable topic? Short, generic or unrelated remarks do NOT count as a topic.
Step 2. If they do not, return exactly: {"insufficient_evidence": true, "reason": "<one short sentence>"}
Step 3. If they do, decide the ov

## The gate

`name_themes` runs the whole sequence and returns one record per theme. The stub
below refuses every group, which is what a real model should do when a group has
no shared topic.

In [3]:
from broadlisten.label import (JUDGE_SYSTEM_PROMPT, name_themes)

def stub(prompts, temperature=0.0, batch_size=None, system=None, max_new=None):
    """Refuses. Fabricates nothing."""
    if system == JUDGE_SYSTEM_PROMPT:
        return ['{"verdict": "no", "reason": "no model was consulted"}'] * len(prompts)
    return ['{"insufficient_evidence": true, '
            '"reason": "no model was consulted"}'] * len(prompts)

records = name_themes(texts, vectors, labels, stub, n_samples=1)
for theme, rec in sorted(records.items()):
    print(f"theme {theme:2d} ({rec['n_members']:3d})  {rec['status']:9s}  "
          f"{rec.get('withheld_reason', '')}")

theme  0 (114)  withheld   no model was consulted
theme  1 ( 93)  withheld   no model was consulted
theme  2 (112)  withheld   no model was consulted
theme  3 (121)  withheld   no model was consulted
theme  4 ( 85)  withheld   no model was consulted


## What a name has to survive

With a real model, a returned name still has to pass all of these. Any one of
them withholds it:

| check | what it asks |
|---|---|
| grounding | are the name's words in the responses it cited? |
| self-consistency | did several samples agree? |
| held-out judge | does the name fit members it was *not* written from? |
| embedding margin | is the name closer to its own theme than to any other? |

The thresholds are `NamingGate`, documented in
[`docs/configuration.md`](../../docs/configuration.md).

In [4]:
import dataclasses
for f in dataclasses.fields(DEFAULT_GATE):
    print(f"{f.name:18s} {getattr(DEFAULT_GATE, f.name)}")

ground_min         0.25
ground_weak        0.4
emb_margin_min     -0.1
emb_margin_weak    0.0
summary_min        0.3
summary_weak       0.5
keypoint_min       0.5
n_representative   14
n_farthest         3
n_judge            8
n_keywords         12
n_samples          3
temperature        0.7
mmr_lambda         0.6


To run this for real, load the model once and pass its `generate`:

```python
from broadlisten.sentiment import load_causal_lm

with load_causal_lm("Qwen/Qwen2.5-14B-Instruct", load_in_4bit=True,
                    max_new_tokens=320) as generate:
    records = name_themes(texts, vectors, labels, generate)
```

**Next:** [6. Sentiment](06_sentiment.ipynb).